---
# 0. Setup

In [ ]:
from pathlib import Path

import holidays as hols
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns



pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_theme(style='whitegrid')

# Notebooks live in EDA/, data lives in <repo root>/data/ (git-ignored -- never commit it).
DATA_DIR = Path('.') / 'data'



---
# 1.  Airport Info Joining (Airport location/specs to county/CBSA)



In [ ]:
import json

from shapely.geometry import shape

# 1. Load County Boundaries CSV
counties_raw = pd.read_csv(
    DATA_DIR /"us-county-boundaries.csv",  
    sep=";",
    dtype={"GEOID": str, "STATEFP": str, "COUNTYFP": str, "CBSAFP": str},
)

# Parse the GeoJSON strings in Geo Shape into Shapely polygons
counties_raw["geometry"] = counties_raw["Geo Shape"].apply(
    lambda x: shape(json.loads(x)) if pd.notnull(x) else None
)

gdf_counties = gpd.GeoDataFrame(
    counties_raw[["GEOID", "CBSAFP", "STATE_NAME", "geometry"]],
    geometry="geometry",
    crs="EPSG:4326",
)

# 2. Load Airport Coordinates CSV
airports = pd.read_csv(DATA_DIR / "AA_AIRPORT_SIMPLE.csv")

gdf_airports = gpd.GeoDataFrame(
    airports,
    geometry=gpd.points_from_xy(airports["LNGTD"], airports["LAT"]),
    crs="EPSG:4326",
)

# 3. Spatial Join: Assign GEOID (County_FIPS) and CBSAFP to every US/PR airport
dim_airports = gpd.sjoin(
    gdf_airports, gdf_counties, how="left", predicate="within"
).drop(columns=["geometry", "index_right"])

dim_airports = dim_airports.rename(
    columns={"GEOID": "County_FIPS", "CBSAFP": "CBSA_Code"}
)

assert dim_airports["AIRPRT_CD"].is_unique, "Duplicate airport codes found!"
print(f"Total airports: {len(dim_airports)}")
print(f"Matched to US/PR County: {dim_airports['County_FIPS'].notna().sum()}")
print(f"Matched to a CBSA: {dim_airports['CBSA_Code'].notna().sum()}")




Total airports: 4565
Matched to US/PR County: 1471
Matched to a CBSA: 1179


In [3]:
dim_airports.sample(10)
dim_airports.isnull().value_counts()


AIRPRT_CD  AIRPRT_NM  CITY_METRO_IATA_CD  STATE_PROVNC_CD  CNTRY_CD  WRLD_AREA_DOT_CD  LNGST_RUNWAY_FT_QTY  ELEVATN_FT_QTY  LAT    LNGTD  County_FIPS  CBSA_Code  STATE_NAME
False      False      False               False            False     False             False                False           False  False  True         True       True          3094
                                                                                                                                          False        False      False         1179
                                                                                                                                                       True       False          292
Name: count, dtype: int64

In [4]:

dim_airports.drop(columns=['STATE_NAME'], axis=1, inplace=True)
dim_airports.drop(columns=['CITY_METRO_IATA_CD'],axis=1,inplace=True)
dim_airports.drop(columns=['AIRPRT_NM','STATE_PROVNC_CD','WRLD_AREA_DOT_CD'], inplace=True)
dim_airports.fillna("MISSING",inplace=True)
dim_airports.head()


,AIRPRT_CD,CNTRY_CD,LNGST_RUNWAY_FT_QTY,ELEVATN_FT_QTY,LAT,LNGTD,County_FIPS,CBSA_Code
0,OAH,AF,7900,3780,33.39,62.26,MISSING,MISSING
1,REX,MX,6200,139,26.01,-98.23,MISSING,MISSING
2,YQX,CA,10200,496,48.94,-54.57,MISSING,MISSING
3,MLW,LR,6000,26,6.29,-10.76,MISSING,MISSING
4,CIJ,BO,8500,805,-11.04,-68.78,MISSING,MISSING


In [ ]:
pd.set_option('display.float_format', '{:,.6f}'.format)
dim_airports.rename(columns={'County_FIPS':'CNTY_FIPS','CBSA_Code':'CBSA_CD'}, inplace=True)
dim_airports.head()

,AIRPRT_CD,CNTRY_CD,LNGST_RUNWAY_FT_QTY,ELEVATN_FT_QTY,LAT,LNGTD,CNTY_FIPS,CBSA_CD
0,OAH,AF,7900,3780,33.392222,62.261111,MISSING,MISSING
1,REX,MX,6200,139,26.008333,-98.228056,MISSING,MISSING
2,YQX,CA,10200,496,48.936944,-54.568056,MISSING,MISSING
3,MLW,LR,6000,26,6.288611,-10.758889,MISSING,MISSING
4,CIJ,BO,8500,805,-11.037778,-68.783333,MISSING,MISSING


In [6]:
dim_airports.to_csv(DATA_DIR / "AIRPORT_V1.csv", index=False)

---
# 2. Forecast info (PAX, REV) + Calendar features

In [7]:


def extend_route_spine(
    df: pd.DataFrame,
    end_year: int,
    end_month: int,
    route_cols: list = ["ORG_AIRPRT", "DES_AIRPRT"],
    year_col: str = "DEP_YEAR",
    month_col: str = "DEP_MONTH",
    fill_zero_cols: list = ["OD_PAX_LOG", "OD_REV_USD_LOG"],
) -> pd.DataFrame:
    """
    Expands a route-month dataframe so every unique route has a continuous row
    from the earliest [Year, Month] in the data up to (end_year, end_month).
    
    Historical gaps (<= max date in df) in `fill_zero_cols` are filled with 0;
    future forecast months (> max date in df) are left as NaN.
    """
    df = df.copy()
    df['year'] = df[year_col]
    df['month'] = df[month_col]
    df.drop(columns=[year_col, month_col], inplace=True)
    year_col = 'year'
    month_col = 'month'
    # 1. Detect the earliest and latest [Year, Month] already in the dataframe
    existing_dates = pd.to_datetime(
        df[[year_col, month_col]].assign(DAY=1)
    )
    min_date = existing_dates.min()
    max_hist_date = existing_dates.max()
    target_end_date = pd.Timestamp(year=end_year, month=end_month, day=1)

    if target_end_date < min_date:
        raise ValueError(f"Target end date ({target_end_date:%Y-%m}) is earlier than min date ({min_date:%Y-%m}).")

    # 2. Extract unique routes and generate the complete monthly date grid
    unique_routes = df[route_cols].drop_duplicates()
    all_dates = pd.date_range(start=min_date, end=target_end_date, freq="MS")
    
    date_grid = pd.DataFrame({
        year_col: all_dates.year,
        month_col: all_dates.month,
        "_temp_date": all_dates,
    })

    # 3. Cross-join routes x dates, then left-join original metrics back
    expanded = unique_routes.merge(date_grid, how="cross")
    expanded = expanded.merge(df, on=route_cols + [year_col, month_col], how="left")

    # 4. Fill missing historical months (<= T) with 0; keep future rows (> T) as NaN
    valid_fill_cols = [c for c in fill_zero_cols if c in expanded.columns]
    if valid_fill_cols:
        hist_mask = expanded["_temp_date"] <= max_hist_date
        expanded.loc[hist_mask, valid_fill_cols] = (
            expanded.loc[hist_mask, valid_fill_cols].fillna(0)
        )

    # 5. Sort cleanly and drop helper date column
    expanded = (
        expanded.sort_values(route_cols + [year_col, month_col])
        .drop(columns=["_temp_date"])
        .reset_index(drop=True)
    )
    expanded['ROUTE'] = expanded[route_cols[0]].astype(str) + '-'+expanded[route_cols[1]].astype(str)
    

    return expanded


In [8]:
base_df = pd.read_csv(DATA_DIR / "made/AA_join.csv")
base_df = base_df[['DEP_MONTH','DEP_YEAR','ORG_AIRPRT','DES_AIRPRT','ROUTE','OD_REV_USD_LOG','OD_PAX_LOG']]
base_df.tail()


,DEP_MONTH,DEP_YEAR,ORG_AIRPRT,DES_AIRPRT,ROUTE,OD_REV_USD_LOG,OD_PAX_LOG
1533517,4,2021,NRT,DFW,NRT-DFW,12.933646,5.328847
1533518,7,2023,NRT,DFW,NRT-DFW,13.573198,6.474091
1533519,10,2025,NRT,DFW,NRT-DFW,14.367177,7.059618
1533520,6,2023,NRT,DFW,NRT-DFW,13.441307,6.358120
1533521,8,2021,NRT,DFW,NRT-DFW,14.869754,7.141190


In [ ]:
# Auto-detect your exact last valid historical date T from raw Passenger/Revenue CSV
full_spine = base_df.copy()
full_spine['ROUTE'] = full_spine['ORG_AIRPRT'].astype(str) + '-' + full_spine['DES_AIRPRT'].astype(str)
full_spine['DATE'] = pd.to_datetime(
    dict(year=full_spine["DEP_YEAR"], month=full_spine["DEP_MONTH"], day=1)
)
raw_max_date = pd.to_datetime(
    dict(year=full_spine["DEP_YEAR"], month=full_spine["DEP_MONTH"], day=1)
).max()
print("Exact historical cutoff T:", raw_max_date.strftime("%Y-%m"))

# fill 0 on or before max date and leave all after as NaN
hist_mask = full_spine["DATE"] <= raw_max_date
full_spine.loc[hist_mask, ["OD_PAX_LOG", "OD_REV_USD_LOG"]] = (
    full_spine.loc[hist_mask, ["OD_PAX_LOG", "OD_REV_USD_LOG"]].fillna(0)
)


Exact historical cutoff T: 2026-07


In [10]:
base_df = extend_route_spine(full_spine,2028,7)
base_df.drop(columns=['DATE'],inplace=True)
base_df.rename(columns={'year':'DEP_YEAR','month':'DEP_MONTH'},inplace=True)
base_df.sample(5)


,ORG_AIRPRT,DES_AIRPRT,DEP_YEAR,DEP_MONTH,ROUTE,OD_REV_USD_LOG,OD_PAX_LOG
823554,LAS,MAN,2022,4,LAS-MAN,13.280973,7.339979
295672,CLE,MSP,2019,8,CLE-MSP,14.390760,9.115416
1038112,MIA,SAT,2019,8,MIA-SAT,13.217064,7.827424
1602043,SDF,MDE,2026,10,SDF-MDE,NaN,NaN
22443,ALB,LIH,2020,7,ALB-LIH,0.000000,0.000000


In [12]:
base_df.isnull().value_counts()

ORG_AIRPRT  DES_AIRPRT  DEP_YEAR  DEP_MONTH  ROUTE  OD_REV_USD_LOG  OD_PAX_LOG
False       False       False     False      False  False           False         1557829
                                                    True            True           410856
Name: count, dtype: int64

In [13]:
base_df.rename(columns={'year':'DEP_YEAR','month':'DEP_MONTH'}, inplace=True)
base_df.groupby(['DEP_YEAR','DEP_MONTH']).agg({'OD_PAX_LOG':'sum','OD_REV_USD_LOG':'sum'}).reset_index().sort_values(['DEP_YEAR','DEP_MONTH'],ascending=True).iloc[-30:]

,DEP_YEAR,DEP_MONTH,OD_PAX_LOG,OD_REV_USD_LOG
85,2026,2,"109,358.260072","208,211.137934"
86,2026,3,"111,086.984552","211,220.599843"
87,2026,4,"111,555.317850","212,473.958561"
88,2026,5,"112,662.578712","212,704.424378"
89,2026,6,"113,897.331185","216,166.606355"
90,2026,7,"115,773.696275","216,457.809005"
91,2026,8,0.000000,0.000000
92,2026,9,0.000000,0.000000
93,2026,10,0.000000,0.000000
94,2026,11,0.000000,0.000000


In [14]:
calendar = pd.read_csv(DATA_DIR / 'calendar_features.csv')
calendar.drop(columns=['is_summer_break','is_winter_holiday'],inplace=True)
calendar.sample(5)

,flight_year,flight_month,WEEKEND,HOLIDAY,EASTER,COVID
137,2028,6,9,1,0,0
112,2026,5,10,1,0,0
6,2017,7,9,1,0,0
82,2023,11,8,3,0,0
54,2021,7,9,2,0,1


In [ ]:
base_df.isnull().value_counts()

ORG_AIRPRT  DES_AIRPRT  DEP_YEAR  DEP_MONTH  ROUTE  OD_REV_USD_LOG  OD_PAX_LOG
False       False       False     False      False  False           False         1557829
                                                    True            True           410856
Name: count, dtype: int64

In [16]:
full_spine = base_df.join(calendar.set_index(['flight_month','flight_year']), on=['DEP_MONTH','DEP_YEAR'], how='left')
full_spine.sort_values(by=['DEP_YEAR','DEP_MONTH']).tail()

,ORG_AIRPRT,DES_AIRPRT,DEP_YEAR,DEP_MONTH,ROUTE,OD_REV_USD_LOG,OD_PAX_LOG,WEEKEND,HOLIDAY,EASTER,COVID
1968224,ZRH,SYR,2028,7,ZRH-SYR,NaN,NaN,9,1,0,0
1968339,ZRH,TPA,2028,7,ZRH-TPA,NaN,NaN,9,1,0,0
1968454,ZRH,TUL,2028,7,ZRH-TUL,NaN,NaN,9,1,0,0
1968569,ZRH,TUS,2028,7,ZRH-TUS,NaN,NaN,9,1,0,0
1968684,ZRH,TYS,2028,7,ZRH-TYS,NaN,NaN,9,1,0,0


# 3. T100 + Base DF

In [17]:
base_df = full_spine.copy()
base_df.head()

,ORG_AIRPRT,DES_AIRPRT,DEP_YEAR,DEP_MONTH,ROUTE,OD_REV_USD_LOG,OD_PAX_LOG,WEEKEND,HOLIDAY,EASTER,COVID
0,ABQ,ALB,2019,1,ABQ-ALB,11.066233,5.544279,8,2,0,0
1,ABQ,ALB,2019,2,ABQ-ALB,10.831547,5.214501,8,1,0,0
2,ABQ,ALB,2019,3,ABQ-ALB,11.714547,5.959432,10,0,0,0
3,ABQ,ALB,2019,4,ABQ-ALB,11.631769,6.113262,8,0,1,0
4,ABQ,ALB,2019,5,ABQ-ALB,11.906085,6.365215,9,1,0,0


In [18]:
T100_2017 = pd.read_csv(DATA_DIR / 'T100_2017.csv')
T100_2018 = pd.read_csv(DATA_DIR / 'T100_2018.csv')
T100_2019 = pd.read_csv(DATA_DIR / 'T100_2019.csv')
T100_2020 = pd.read_csv(DATA_DIR / 'T100_2020.csv')
T100_2021 = pd.read_csv(DATA_DIR / 'T100_2021.csv')
T100_2022 = pd.read_csv(DATA_DIR / 'T100_2022.csv')
T100_2023 = pd.read_csv(DATA_DIR / 'T100_2023.csv')
T100_2024 = pd.read_csv(DATA_DIR / 'T100_2024.csv')
T100_2025 = pd.read_csv(DATA_DIR / 'T100_2025.csv')
T100_2026 = pd.read_csv(DATA_DIR / 'T100_2026.csv')
T100 = pd.concat([T100_2017,T100_2018, T100_2019, T100_2020, T100_2021, T100_2022, T100_2023, T100_2024, T100_2025, T100_2026], ignore_index=True)


In [19]:
T100.to_csv(DATA_DIR / 'T100.csv', index=False)


In [20]:
T100.sort_values(by=['YEAR','MONTH']).tail()

,PASSENGERS,FREIGHT,MAIL,UNIQUE_CARRIER,ORIGIN_AIRPORT_ID,ORIGIN,DEST_AIRPORT_ID,DEST,YEAR,MONTH
2752456,"72,704.000000","934,467.000000","111,190.000000",UA,11618,EWR,14771,SFO,2026,8
2752458,"73,281.000000","3,830,967.000000","111,710.000000",UA,14771,SFO,11618,EWR,2026,8
2752465,"77,757.000000","45,539.000000",0.000000,DL,13204,MCO,10397,ATL,2026,8
2752466,"78,154.000000","91,572.000000",0.000000,DL,10397,ATL,13204,MCO,2026,8
2752468,"78,492.000000","407,439.000000","59,774.000000",AS,10299,ANC,14747,SEA,2026,8


In [ ]:

def create_grid_lags(
    df: pd.DataFrame,
    entity_cols: list,
    value_cols: list,
    lags: list = [12, 24],
    max_target_date: str = "2028-12-31",
    year_col: str = "Year",
    month_col: str = "Month",
    drop_raw: bool = True,
) -> pd.DataFrame:
    # Uses `df`, `year_col`, and `month_col` to build a proper timestamp
    work_df = df.copy()
    work_df["_date"] = pd.to_datetime(
        dict(year=work_df[year_col], month=work_df[month_col], day=1)
    )

    min_date = work_df["_date"].min()
    max_hist_date = work_df["_date"].max()
    # Uses `max_target_date` to set the end of the monthly calendar
    end_date = pd.to_datetime(max_target_date)

    # Uses `year_col` and `month_col` to build the complete monthly date grid
    all_dates = pd.date_range(start=min_date, end=end_date, freq="MS")
    date_grid = pd.DataFrame({
        year_col: all_dates.year,
        month_col: all_dates.month,
        "_date": all_dates,
    })

    # Initializes `grid` by cross-joining unique entities with every month in `date_grid`
    entities = work_df.loc[:, entity_cols].drop_duplicates()
    join_keys = list(entity_cols) + [year_col, month_col]
    
    grid = entities.merge(date_grid, how="cross").merge(
        work_df.drop(columns=["_date"]),
        on=join_keys,
        how="left",
    )

    # Fills missing historical months (<= max date in `df`) with 0 and keeps future rows (> T) as NaN
    hist_mask = grid["_date"] <= max_hist_date
    grid.loc[hist_mask, value_cols] = grid.loc[hist_mask, value_cols].fillna(0)

    # Sorts `grid` chronologically and shifts `value_cols` by each lag in `lags`
    sort_keys = list(entity_cols) + ["_date"]
    grid = grid.sort_values(sort_keys).reset_index(drop=True)

    for col in value_cols:
        for l in lags:
            grid[f"{col}_lag_{l}"] = grid.groupby(entity_cols)[col].shift(l)

    cols_to_drop = ["_date"] + (list(value_cols) if drop_raw else list())
    return grid.drop(columns=cols_to_drop)


In [ ]:
def left_join_feature_table(
    spine: pd.DataFrame,
    feature_df: pd.DataFrame,
    left_keys: list,
    right_keys: list = None,
    prefixes: list = None,
) -> pd.DataFrame:
  """
    Left-joins `feature_df` onto `spine` while strictly preserving row count.
    - For a single join : leave `prefixes=None`.
    - For a double join : pass `left_keys=[["Origin", "Year", "Month"], ["Destination", "Year", "Month"]]`
      and `prefixes=["ORG_", "DES_"]`.
    """
  rows_before = len(spine)
  right_keys = right_keys or left_keys

  # Case A: Single Left-Join
  if prefixes is None:
    spine = spine.merge(
        feature_df,
        left_on=left_keys,
        right_on=right_keys,
        how="left",
    )
    # Drop duplicate right-hand join keys if names differed
    extra_keys = [k for k in right_keys if k not in left_keys]
    if extra_keys:
      spine = spine.drop(columns=extra_keys)

  # Case B: Multi/Double Left-Join 
    for l_keys, prefix in zip(left_keys, prefixes):
      prefixed_df = feature_df.add_prefix(prefix)
      prefixed_r_keys = [f"{prefix}{k}" for k in right_keys]

      spine = spine.merge(
          prefixed_df,
          left_on=l_keys,
          right_on=prefixed_r_keys,
          how="left",
      ).drop(columns=prefixed_r_keys)

  assert (
      len(spine) == rows_before
  ), f"Row count exploded! {rows_before} -> {len(spine)}"
  return spine


In [ ]:

t100 = T100.copy()
target_carrier = "AA"  
t100["route"] = (
    t100["ORIGIN"].astype(str).str.strip()
    + "-"
    + t100["DEST"].astype(str).str.strip()
)

# Aggregate Total Market Volume (All Carriers)
market_t100 = t100.groupby(["route", "YEAR", "MONTH"], as_index=False).agg(
    T100_Market_Pax=("PASSENGERS", "sum"),
    T100_Market_Freight=("FREIGHT", "sum"),
    T100_Carrier_Count=("UNIQUE_CARRIER", "nunique"),
)

# Aggregate Target Carrier Volume 
carrier_t100 = (
    t100[t100["UNIQUE_CARRIER"] == target_carrier]
    .groupby(["route", "YEAR", "MONTH"], as_index=False)
    .agg(T100_Target_Pax=("PASSENGERS", "sum"))
)

# Combine & Compute Competitive Market Share
t100_agg = market_t100.merge(
    carrier_t100, on=["route", "YEAR", "MONTH"], how="left"
).fillna({"T100_Target_Pax": 0})

t100_agg["T100_Target_Pax_Share"] = (
    t100_agg["T100_Target_Pax"] / t100_agg["T100_Market_Pax"].replace(0, pd.NA)
).fillna(0)

t100_agg = t100_agg.rename(columns={"YEAR": "Year", "MONTH": "Month"})

# Run your general lag function 
t100_lagged = create_grid_lags(
    df=t100_agg,
    entity_cols=["route"],
    value_cols=["T100_Market_Pax", "T100_Target_Pax_Share", "T100_Carrier_Count"],
    lags=(12, 24),
    max_target_date="2028-07-01",
    year_col="Year",
    month_col="Month",
    drop_raw=True,
)



C:\Users\danie\AppData\Local\Temp\ipykernel_61040\1412025964.py:31: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0)


In [24]:
t100_lagged.sort_values(by=['Year','Month']).tail()

,route,Year,Month,T100_Market_Freight,T100_Target_Pax,T100_Market_Pax_lag_12,T100_Market_Pax_lag_24,T100_Target_Pax_Share_lag_12,T100_Target_Pax_Share_lag_24,T100_Carrier_Count_lag_12,T100_Carrier_Count_lag_24
13485779,ZXU-WWD,2028,7,NaN,NaN,NaN,0.000000,NaN,0.000000,NaN,0.000000
13485918,ZXU-YFC,2028,7,NaN,NaN,NaN,0.000000,NaN,0.000000,NaN,0.000000
13486057,ZZV-FID,2028,7,NaN,NaN,NaN,0.000000,NaN,0.000000,NaN,0.000000
13486196,ZZV-SAT,2028,7,NaN,NaN,NaN,0.000000,NaN,0.000000,NaN,0.000000
13486335,ZZV-SDF,2028,7,NaN,NaN,NaN,0.000000,NaN,0.000000,NaN,0.000000


In [25]:

full_spine = left_join_feature_table(
    spine=base_df,
    feature_df=t100_lagged,
    left_keys=["ROUTE", "DEP_YEAR", "DEP_MONTH"],
    right_keys=["route", "Year", "Month"]
)

In [26]:
full_spine.head()

,ORG_AIRPRT,DES_AIRPRT,DEP_YEAR,DEP_MONTH,ROUTE,OD_REV_USD_LOG,OD_PAX_LOG,WEEKEND,HOLIDAY,EASTER,COVID,T100_Market_Freight,T100_Target_Pax,T100_Market_Pax_lag_12,T100_Market_Pax_lag_24,T100_Target_Pax_Share_lag_12,T100_Target_Pax_Share_lag_24,T100_Carrier_Count_lag_12,T100_Carrier_Count_lag_24
0,ABQ,ALB,2019,1,ABQ-ALB,11.066233,5.544279,8,2,0,0,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,ABQ,ALB,2019,2,ABQ-ALB,10.831547,5.214501,8,1,0,0,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,ABQ,ALB,2019,3,ABQ-ALB,11.714547,5.959432,10,0,0,0,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,ABQ,ALB,2019,4,ABQ-ALB,11.631769,6.113262,8,0,1,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,ABQ,ALB,2019,5,ABQ-ALB,11.906085,6.365215,9,1,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [27]:
full_spine.drop(columns=['route','T100_Market_Freight','T100_Target_Pax'],inplace=True, errors='ignore')
for col in full_spine.columns:
    print(full_spine[col].isnull().value_counts())

ORG_AIRPRT
False    1968685
Name: count, dtype: int64
DES_AIRPRT
False    1968685
Name: count, dtype: int64
DEP_YEAR
False    1968685
Name: count, dtype: int64
DEP_MONTH
False    1968685
Name: count, dtype: int64
ROUTE
False    1968685
Name: count, dtype: int64
OD_REV_USD_LOG
False    1557829
True      410856
Name: count, dtype: int64
OD_PAX_LOG
False    1557829
True      410856
Name: count, dtype: int64
WEEKEND
False    1968685
Name: count, dtype: int64
HOLIDAY
False    1968685
Name: count, dtype: int64
EASTER
False    1968685
Name: count, dtype: int64
COVID
False    1968685
Name: count, dtype: int64
T100_Market_Pax_lag_12
False    1310504
True      658181
Name: count, dtype: int64
T100_Market_Pax_lag_24
False    1449115
True      519570
Name: count, dtype: int64
T100_Target_Pax_Share_lag_12
False    1310504
True      658181
Name: count, dtype: int64
T100_Target_Pax_Share_lag_24
False    1449115
True      519570
Name: count, dtype: int64
T100_Carrier_Count_lag_12
False    1310504
True

In [28]:

unmatched_routes = full_spine[full_spine["T100_Market_Pax_lag_12"].isna()]["ROUTE"].unique()
print(f"Total unmatched routes: {len(unmatched_routes):,}")
print("First 20 unmatched routes:", unmatched_routes[100:150:3])



Total unmatched routes: 17,119
First 20 unmatched routes: ['ABQ-PTY' 'ABQ-PVR' 'ABQ-RIC' 'ABQ-RSW' 'ABQ-SAT' 'ABQ-SDQ' 'ABQ-SJC'
 'ABQ-SJU' 'ABQ-SNA' 'ABQ-SYR' 'ABQ-TPE' 'ABQ-TYS' 'ABQ-YUL' 'ABQ-YYZ'
 'ALB-AMS' 'ALB-AUA' 'ALB-BNA']


In [29]:
full_spine.ROUTE.isnull().value_counts()
full_spine.head()

,ORG_AIRPRT,DES_AIRPRT,DEP_YEAR,DEP_MONTH,ROUTE,OD_REV_USD_LOG,OD_PAX_LOG,WEEKEND,HOLIDAY,EASTER,COVID,T100_Market_Pax_lag_12,T100_Market_Pax_lag_24,T100_Target_Pax_Share_lag_12,T100_Target_Pax_Share_lag_24,T100_Carrier_Count_lag_12,T100_Carrier_Count_lag_24
0,ABQ,ALB,2019,1,ABQ-ALB,11.066233,5.544279,8,2,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,ABQ,ALB,2019,2,ABQ-ALB,10.831547,5.214501,8,1,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,ABQ,ALB,2019,3,ABQ-ALB,11.714547,5.959432,10,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,ABQ,ALB,2019,4,ABQ-ALB,11.631769,6.113262,8,0,1,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,ABQ,ALB,2019,5,ABQ-ALB,11.906085,6.365215,9,1,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [30]:
full_spine.to_csv(DATA_DIR / "made/AA_T100.csv", index=False)

In [31]:
t100_routes = set(t100["route"].unique())
matched_spine = full_spine[(full_spine["ROUTE"].isin(t100_routes)) & (full_spine["DEP_YEAR"] <= 2026)]
print("Null % on matched T-100 routes (2019-2026):", matched_spine["T100_Market_Pax_lag_12"].isna().mean() * 100)


Null % on matched T-100 routes (2019-2026): 0.0


In [ ]:
# Get the exact set of unique routes in spine vs. raw T-100 CSV
spine_routes = set(full_spine["ROUTE"].unique())
t100_routes = set(t100["route"].unique())

missing_from_t100 = spine_routes - t100_routes
in_both = spine_routes & t100_routes

print(f"Total spine routes: {len(spine_routes):,}")
print(f"Routes in T-100:    {len(in_both):,} ({len(in_both)/len(spine_routes)*100:.1f}%)")
print(f"Missing from T-100: {len(missing_from_t100):,} ({len(missing_from_t100)/len(spine_routes)*100:.1f}%)")

# Verify that among routes that ARE in T-100, there are 0.0% nulls across 2019-2026
matched_hist = full_spine[(full_spine["ROUTE"].isin(in_both)) & (full_spine["DEP_YEAR"] <= 2026)]
print("\nNull % on routes that exist in T-100 (2019-2026):")
print(matched_hist[["T100_Market_Pax_lag_12", "T100_Market_Pax_lag_24"]].isna().mean() * 100)

# Verify that among routes not in T-100, 100.0% of rows are null 
unmatched_all = full_spine[full_spine["ROUTE"].isin(missing_from_t100)]
print("\nNull % on routes missing from T-100:")
print(unmatched_all[["T100_Market_Pax_lag_12", "T100_Market_Pax_lag_24"]].isna().mean() * 100)


Total spine routes: 17,119
Routes in T-100:    12,601 (73.6%)
Missing from T-100: 4,518 (26.4%)

Null % on routes that exist in T-100 (2019-2026):
T100_Market_Pax_lag_12   0.000000
T100_Market_Pax_lag_24   0.000000
dtype: float64

Null % on routes missing from T-100:
T100_Market_Pax_lag_12   100.000000
T100_Market_Pax_lag_24   100.000000
dtype: float64


In [33]:
base_df = full_spine.copy()
base_df.head()

,ORG_AIRPRT,DES_AIRPRT,DEP_YEAR,DEP_MONTH,ROUTE,OD_REV_USD_LOG,OD_PAX_LOG,WEEKEND,HOLIDAY,EASTER,COVID,T100_Market_Pax_lag_12,T100_Market_Pax_lag_24,T100_Target_Pax_Share_lag_12,T100_Target_Pax_Share_lag_24,T100_Carrier_Count_lag_12,T100_Carrier_Count_lag_24
0,ABQ,ALB,2019,1,ABQ-ALB,11.066233,5.544279,8,2,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,ABQ,ALB,2019,2,ABQ-ALB,10.831547,5.214501,8,1,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,ABQ,ALB,2019,3,ABQ-ALB,11.714547,5.959432,10,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,ABQ,ALB,2019,4,ABQ-ALB,11.631769,6.113262,8,0,1,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,ABQ,ALB,2019,5,ABQ-ALB,11.906085,6.365215,9,1,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [34]:
base_df.rename(columns={'weekend_peak_days':'WEEKEND', 'us_holiday_count':'HOLIDAY',
       'is_easter_month':'EASTER', 'is_covid':'COVID', 'T100_Market_Pax_lag_12':'MPAX_L12',
       'T100_Market_Pax_lag_24':'MPAX_L24', 'T100_Target_Pax_Share_lag_12':'PAX_SHARE_L12',
       'T100_Target_Pax_Share_lag_24':'PAX_SHARE_L24', 'T100_Carrier_Count_lag_12':'AIRLN_COUNT_L12',
       'T100_Carrier_Count_lag_24':'AIRLN_COUNT_L24'},inplace=True)
base_df.head()

,ORG_AIRPRT,DES_AIRPRT,DEP_YEAR,DEP_MONTH,ROUTE,OD_REV_USD_LOG,OD_PAX_LOG,WEEKEND,HOLIDAY,EASTER,COVID,MPAX_L12,MPAX_L24,PAX_SHARE_L12,PAX_SHARE_L24,AIRLN_COUNT_L12,AIRLN_COUNT_L24
0,ABQ,ALB,2019,1,ABQ-ALB,11.066233,5.544279,8,2,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,ABQ,ALB,2019,2,ABQ-ALB,10.831547,5.214501,8,1,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,ABQ,ALB,2019,3,ABQ-ALB,11.714547,5.959432,10,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,ABQ,ALB,2019,4,ABQ-ALB,11.631769,6.113262,8,0,1,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,ABQ,ALB,2019,5,ABQ-ALB,11.906085,6.365215,9,1,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [35]:
base_df = base_df.drop(columns=[col for col in base_df.columns if col.endswith('12')])

base_df[(base_df.DEP_YEAR == 2027) & (base_df.DEP_MONTH >1)].sample(5)

,ORG_AIRPRT,DES_AIRPRT,DEP_YEAR,DEP_MONTH,ROUTE,OD_REV_USD_LOG,OD_PAX_LOG,WEEKEND,HOLIDAY,EASTER,COVID,MPAX_L24,PAX_SHARE_L24,AIRLN_COUNT_L24
1017734,MEX,GRR,2027,4,MEX-GRR,NaN,NaN,9,0,0,0,NaN,NaN,NaN
415600,DEN,ANC,2027,10,DEN-ANC,NaN,NaN,10,1,0,0,"3,530.000000",0.000000,1.000000
1449214,PWM,MKE,2027,4,PWM-MKE,NaN,NaN,9,0,0,0,0.000000,0.000000,0.000000
538760,FAT,NRT,2027,5,FAT-NRT,NaN,NaN,9,1,0,0,NaN,NaN,NaN
210663,BUF,MAD,2027,3,BUF-MAD,NaN,NaN,8,0,1,0,NaN,NaN,NaN


In [85]:
base_df.sort_values(by='AIRLN_COUNT_L24',ascending=False)

,ORG_AIRPRT,DES_AIRPRT,DEP_YEAR,DEP_MONTH,ROUTE,OD_REV_USD_LOG,OD_PAX_LOG,WEEKEND,HOLIDAY,EASTER,COVID,MPAX_L24,PAX_SHARE_L24,AIRLN_COUNT_L24,ORG_CARGO_WGT,ORG_SVC_LVL,ORG_HUB,DES_CARGO_WGT,DES_SVC_LVL,DES_HUB,Cargo_Lookup_Year
165370,BOG,MIA,2019,1,BOG-MIA,15.05,9.72,8,2,0,0,"30,299.00",0.24,16.00,NaN,NaN,NaN,NaN,NaN,NaN,2017
845831,LAX,SEA,2019,7,LAX-SEA,16.21,11.32,8,1,0,0,"140,819.00",0.06,16.00,NaN,NaN,NaN,NaN,NaN,NaN,2017
165371,BOG,MIA,2019,2,BOG-MIA,14.69,9.66,8,1,0,0,"22,842.00",0.26,16.00,NaN,NaN,NaN,NaN,NaN,NaN,2017
1026376,MIA,BOG,2019,2,MIA-BOG,14.66,9.62,8,1,0,0,"22,716.00",0.26,15.00,NaN,NaN,NaN,NaN,NaN,NaN,2017
1026375,MIA,BOG,2019,1,MIA-BOG,15.37,10.07,8,2,0,0,"28,273.00",0.25,15.00,NaN,NaN,NaN,NaN,NaN,NaN,2017
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1968680,ZRH,TYS,2028,3,ZRH-TYS,NaN,NaN,9,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,"297,843,482.00",P,S,2020
1968681,ZRH,TYS,2028,4,ZRH-TYS,NaN,NaN,9,0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,"297,843,482.00",P,S,2020
1968682,ZRH,TYS,2028,5,ZRH-TYS,NaN,NaN,8,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,"297,843,482.00",P,S,2020
1968683,ZRH,TYS,2028,6,ZRH-TYS,NaN,NaN,9,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,"297,843,482.00",P,S,2020


In [36]:
base_df.to_csv(DATA_DIR/'made/AA_MASTER_V1.csv',index=False)

---
# 3 Base_df + cargo + enplanements

## Cargo

In [ ]:

years = range(2017, 2026)
cargo_dfs = []

for year in years:
  df = pd.read_csv(DATA_DIR / f'Cargo{year}.csv')

  df.columns = df.columns.str.strip()

  # Find and match the column containing the current year and 'Landed Weight'
  weight_col = [
      col
      for col in df.columns
      if str(year) in col and (f'{year} landed\nweight (lbs.)' in col.lower() or f'{year} landed\r\nweight (lbs.)' in col.lower())
  ][0]

  # Rename to standard column name and tag the year
  df = df.rename(columns={weight_col: 'LANDED_WEIGHT'})
  df['YEAR'] = year

  

  df.drop(columns=[f'{year} Landed\nWeight (lbs.)', f'{year-1} Landed\nWeight (lbs.)', '% Change'], inplace=True, errors='ignore')
  cargo_dfs.append(df)


# Combine all years into one master DataFrame
Cargo = pd.concat(cargo_dfs, ignore_index=True)
Cargo.head()


,Rank,RO,ADO,ST,Locid,Airport Name,City,Svc Lvl,Hub,LANDED_WEIGHT,YEAR,2019 Landed\r\nWeight (lbs.),No.
0,1,SO,MEM,TN,MEM,Memphis International,Memphis,P,S,"23,949,525,780.000000",2017,NaN,NaN
1,2,AL,AAL,AK,ANC,Ted Stevens Anchorage International,Anchorage,P,M,"17,337,337,377.000000",2017,NaN,NaN
2,3,SO,MEM,KY,SDF,Louisville International-Standiford Field,Louisville,P,S,"13,403,682,652.000000",2017,NaN,NaN
3,4,GL,CHI,IL,ORD,Chicago O'Hare International,Chicago,P,L,"10,373,559,593.000000",2017,NaN,NaN
4,5,SO,ORL,FL,MIA,Miami International,Miami,P,L,"7,963,988,407.000000",2017,NaN,NaN


In [38]:
Cargo.drop(columns=['ST','Rank','ADO','RO','Airport Name','City','No.',
'FAA ADO','FAA Region'],inplace=True, errors='ignore')
print(Cargo.isnull().value_counts())
Cargo.head()


Locid  Svc Lvl  Hub    LANDED_WEIGHT  YEAR   2019 Landed\r\nWeight (lbs.)
False  False    False  False          False  True                            1162
                                             False                            144
True   True     True   False          False  False                              1
                                             True                               1
                       True           False  True                               1
Name: count, dtype: int64


,Locid,Svc Lvl,Hub,LANDED_WEIGHT,YEAR,2019 Landed\r\nWeight (lbs.)
0,MEM,P,S,"23,949,525,780.000000",2017,NaN
1,ANC,P,M,"17,337,337,377.000000",2017,NaN
2,SDF,P,S,"13,403,682,652.000000",2017,NaN
3,ORD,P,L,"10,373,559,593.000000",2017,NaN
4,MIA,P,L,"7,963,988,407.000000",2017,NaN


In [39]:
Cargo.dropna(inplace=True)
print(Cargo.isnull().value_counts()) 
print(Cargo[Cargo.isnull().any(axis=1)])


Locid  Svc Lvl  Hub    LANDED_WEIGHT  YEAR   2019 Landed\r\nWeight (lbs.)
False  False    False  False          False  False                           144
Name: count, dtype: int64
Empty DataFrame
Columns: [Locid, Svc Lvl, Hub, LANDED_WEIGHT, YEAR, 2019 Landed
Weight (lbs.)]
Index: []


In [40]:
Cargo.rename(columns={'Locid':'LOCID','Svc Lvl':'SVC_LVL','Hub':'HUB'},inplace=True)
Cargo.to_csv(DATA_DIR / 'made/Cargo.csv', index=False)


In [ ]:

# Standardize column names & ensure strictly 1 row per ['Airport', 'Year']
cargo_clean = (
    Cargo.rename(columns={
        "LOCID": "Airport",
        "YEAR": "Year",
        "LANDED_WEIGHT": "Cargo_Landed_Weight",
        "SVC_LVL": "Airport_Svc_Level",
        "HUB": "Airport_Hub_Size",
    })
    .sort_values("Cargo_Landed_Weight", ascending=False)
    .groupby(["Airport", "Year"], as_index=False)
    .agg(
        Cargo_Landed_Weight=("Cargo_Landed_Weight", "sum"),
        Airport_Svc_Level=("Airport_Svc_Level", "first"),
        Airport_Hub_Size=("Airport_Hub_Size", "first"),
    )
)

# Lag Landed Weight by 1 and 2 years 
# Build a continuous yearly grid per Airport out to 2028 so 2027-2028 get lagged values
all_years = pd.DataFrame({"Year": range(cargo_clean["Year"].min(), 2029)})
airports = cargo_clean[["Airport"]].drop_duplicates()

cargo_grid = airports.merge(all_years, how="cross").merge(
    cargo_clean, on=["Airport", "Year"], how="left"
)

# Fill historical missing years (<= max year in cargo_df) with 0
max_cargo_year = cargo_clean["Year"].max()
cargo_grid.loc[cargo_grid["Year"] <= max_cargo_year, "Cargo_Landed_Weight"] = (
    cargo_grid.loc[cargo_grid["Year"] <= max_cargo_year, "Cargo_Landed_Weight"].fillna(0)
)

# Forward-fill static categorical tiers (svc level & hub size) into 2027-2028
cargo_grid = cargo_grid.sort_values(["Airport", "Year"]).reset_index(drop=True)
cargo_grid[["Airport_Svc_Level", "Airport_Hub_Size"]] = (
    cargo_grid.groupby("Airport")[["Airport_Svc_Level", "Airport_Hub_Size"]].ffill()
)

# Create 1-year and 2-year lags of Landed Weight, then drop raw unlagged weight
cargo_grid["Cargo_Landed_Weight_lag_1yr"] = cargo_grid.groupby("Airport")["Cargo_Landed_Weight"].shift(1)
cargo_grid["Cargo_Landed_Weight_lag_2yr"] = cargo_grid.groupby("Airport")["Cargo_Landed_Weight"].shift(2)
cargo_ready = cargo_grid.drop(columns=["Cargo_Landed_Weight"])
cargo_ready.sample(5)



,Airport,Year,Airport_Svc_Level,Airport_Hub_Size,Cargo_Landed_Weight_lag_1yr,Cargo_Landed_Weight_lag_2yr
1096,SCK,2027,P,N,NaN,NaN
8,ABE,2028,P,N,NaN,NaN
1192,STL,2024,P,M,NaN,NaN
1237,TOL,2024,P,N,NaN,NaN
176,BQN,2025,P,N,NaN,NaN


In [42]:
full_spine = base_df.copy()
full_spine.head()


,ORG_AIRPRT,DES_AIRPRT,DEP_YEAR,DEP_MONTH,ROUTE,OD_REV_USD_LOG,OD_PAX_LOG,WEEKEND,HOLIDAY,EASTER,COVID,MPAX_L24,PAX_SHARE_L24,AIRLN_COUNT_L24
0,ABQ,ALB,2019,1,ABQ-ALB,11.066233,5.544279,8,2,0,0,0.000000,0.000000,0.000000
1,ABQ,ALB,2019,2,ABQ-ALB,10.831547,5.214501,8,1,0,0,0.000000,0.000000,0.000000
2,ABQ,ALB,2019,3,ABQ-ALB,11.714547,5.959432,10,0,0,0,0.000000,0.000000,0.000000
3,ABQ,ALB,2019,4,ABQ-ALB,11.631769,6.113262,8,0,1,0,0.000000,0.000000,0.000000
4,ABQ,ALB,2019,5,ABQ-ALB,11.906085,6.365215,9,1,0,0,0.000000,0.000000,0.000000


In [ ]:



latest_cargo_year = int(cargo_clean["Year"].max())  # 2025
full_spine["Cargo_Lookup_Year"] = (full_spine["DEP_YEAR"] - 2).clip(upper=latest_cargo_year)
cargo_clean.rename(columns={
        "Cargo_Landed_Weight": "CARGO_WGT",
        "Airport_Svc_Level": "SVC_LVL",
        "Airport_Hub_Size": "HUB",
    },inplace=True)

# Double Left-Join cargo_clean directly onto full_spine 
full_spine1 = left_join_feature_table(
    spine=full_spine,
    feature_df=cargo_clean,
    left_keys=[["ORG_AIRPRT", "Cargo_Lookup_Year"], ["DES_AIRPRT", "Cargo_Lookup_Year"]],
    right_keys=["Airport", "Year"],
    prefixes=["ORG_", "DES_"],
)

full_spine1 = full_spine1.drop(columns=["Cargo_Lookup_Year"])


In [44]:
pd.set_option('display.float_format', '{:,.2f}'.format)
full_spine1.head()

,ORG_AIRPRT,DES_AIRPRT,DEP_YEAR,DEP_MONTH,ROUTE,OD_REV_USD_LOG,OD_PAX_LOG,WEEKEND,HOLIDAY,EASTER,COVID,MPAX_L24,PAX_SHARE_L24,AIRLN_COUNT_L24,ORG_CARGO_WGT,ORG_SVC_LVL,ORG_HUB,DES_CARGO_WGT,DES_SVC_LVL,DES_HUB
0,ABQ,ALB,2019,1,ABQ-ALB,11.07,5.54,8,2,0,0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN
1,ABQ,ALB,2019,2,ABQ-ALB,10.83,5.21,8,1,0,0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN
2,ABQ,ALB,2019,3,ABQ-ALB,11.71,5.96,10,0,0,0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN
3,ABQ,ALB,2019,4,ABQ-ALB,11.63,6.11,8,0,1,0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN
4,ABQ,ALB,2019,5,ABQ-ALB,11.91,6.37,9,1,0,0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN


In [45]:
full_spine1.loc[:, full_spine1.columns.str.endswith('WGT')].isnull().value_counts()


ORG_CARGO_WGT  DES_CARGO_WGT
True           True             722697
False          False            491538
True           False            380701
False          True             373749
Name: count, dtype: int64

In [46]:
full_spine1.loc[:, full_spine1.columns.str.endswith('LVL')].isnull().value_counts()


ORG_SVC_LVL  DES_SVC_LVL
True         True           722697
False        False          491538
True         False          380701
False        True           373749
Name: count, dtype: int64

In [47]:
full_spine1.loc[:, full_spine1.columns.str.endswith('HUB')].isnull().value_counts()


ORG_HUB  DES_HUB
True     True       722697
False    False      491538
True     False      380701
False    True       373749
Name: count, dtype: int64

In [48]:
base_df = full_spine1
base_df.head()


,ORG_AIRPRT,DES_AIRPRT,DEP_YEAR,DEP_MONTH,ROUTE,OD_REV_USD_LOG,OD_PAX_LOG,WEEKEND,HOLIDAY,EASTER,COVID,MPAX_L24,PAX_SHARE_L24,AIRLN_COUNT_L24,ORG_CARGO_WGT,ORG_SVC_LVL,ORG_HUB,DES_CARGO_WGT,DES_SVC_LVL,DES_HUB
0,ABQ,ALB,2019,1,ABQ-ALB,11.07,5.54,8,2,0,0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN
1,ABQ,ALB,2019,2,ABQ-ALB,10.83,5.21,8,1,0,0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN
2,ABQ,ALB,2019,3,ABQ-ALB,11.71,5.96,10,0,0,0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN
3,ABQ,ALB,2019,4,ABQ-ALB,11.63,6.11,8,0,1,0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN
4,ABQ,ALB,2019,5,ABQ-ALB,11.91,6.37,9,1,0,0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN


In [49]:
base_df.to_csv(DATA_DIR / 'made/AA_MASTER_V2.csv')

To be Continued in part 2...
